# Kartutsnitt

In [1]:
import streamlit as st
import pandas as pd
import pydeck as pdk
import geopandas as gpd
import polars as pl
import os
import fiona
from dotenv import load_dotenv

In [2]:
crs_plot = "EPSG:4326"
crs_norge = "EPSG:25833"
load_dotenv()

True

In [3]:
gdf = gpd.read_file("buffrede_sentrumssoner.gpkg", engine="pyogrio")
gdf["name"] = gdf["område"].astype(str) + " - " + gdf["kommunenavn"].astype(str)
gdf = gdf.to_crs(crs_plot)
MAPBOX_TOKEN = os.getenv("MAPBOX_API_KEY")

In [4]:
områder_path = "geografiske_områder/"
område_gdf_list = []

for root, dirs, files in os.walk(områder_path):
    for f in files:
        if f.endswith("geojson"):
            l = fiona.listlayers(f"{root}/{f}")
            ll = [f for f in l if f.lower().endswith("kommune")]
            if len(ll)==1:
                ll = ll[0]
                område_gdf_list.append(gpd.read_file(f"{root}/{f}", layer=ll).assign(område = root.removeprefix("geografiske_områder/"),
                                                                                     kommune = f))
            else:
                print(f"problem med {root}/{f} og layers {l}. finn på noe smartere!")
områder_gdf = pd.concat([gdf.to_crs(crs_norge) for gdf in område_gdf_list], ignore_index=True)

områder_gdf = områder_gdf[["område", "navn", "geometry", "kommune"]]
navneliste=pl.from_pandas(områder_gdf.drop(columns=["geometry"])).select(pl.col("navn").list.get(0).struct.field("navn")).get_column("navn").to_list()
områder_gdf["kommunenavn"] = navneliste

områder_gdf

,område,navn,geometry,kommune,kommunenavn
0,Bergensområdet,"[{'navn': 'Bergen', 'rekkefolge': '', 'sprak':...","POLYGON ((-40889.251 6732715.54, -41185.821 67...",Basisdata_1201_Bergen_25832_Kommuner2019_GeoJS...,Bergen
1,Bergensområdet,"[{'navn': 'Os', 'rekkefolge': '', 'sprak': 'no...","POLYGON ((-39434.611 6710972.25, -36231.251 67...",Basisdata_1243_Os_25832_Kommuner2019_GeoJSON.g...,Os
2,Bergensområdet,"[{'navn': 'Fjell', 'rekkefolge': '', 'sprak': ...","POLYGON ((-81178.221 6744231.53, -80969.191 67...",Basisdata_1246_Fjell_25832_Kommuner2019_GeoJSO...,Fjell
3,Bergensområdet,"[{'navn': 'Askøy', 'rekkefolge': '', 'sprak': ...","POLYGON ((-48466.041 6746207.53, -46259.711 67...",Basisdata_1247_Askoy_25832_Kommuner2019_GeoJSO...,Askøy
4,Bergensområdet,"[{'navn': 'Lindås', 'rekkefolge': '', 'sprak':...","POLYGON ((-38603.821 6764433.21, -35414.481 67...",Basisdata_1263_Lindas_25832_Kommuner2019_GeoJS...,Lindås
5,Buskerudbyen,NaN,"MULTIPOLYGON (((238078.346 6609766.134, 238093...",Basisdata_3301_Drammen_25832_Kommune_GeoJSON.g...,NaN
6,Buskerudbyen,NaN,"MULTIPOLYGON (((199851.748 6629947.834, 199852...",Basisdata_3303_Kongsberg_25832_Kommune_GeoJSON...,NaN
7,Buskerudbyen,NaN,"MULTIPOLYGON (((233581.941 6631686.23, 234154....",Basisdata_3312_Lier_25832_Kommune_GeoJSON.geojson,NaN
8,Buskerudbyen,NaN,"MULTIPOLYGON (((203676.642 6639124.935, 203677...",Basisdata_3314_Ovre_Eiker_25832_Kommune_GeoJSO...,NaN
9,Grenland,NaN,"MULTIPOLYGON (((194619.667 6569622.081, 194616...",Basisdata_4001_Porsgrunn_25832_Kommune_GeoJSON...,NaN


In [5]:
# 2. Last Parquet-filer
df_7_18 = pd.read_parquet("stasjoner_med_frekvens_10_15_7_18.parquet")

gdf_7_18 = gpd.GeoDataFrame(
    df_7_18,
    geometry=gpd.points_from_xy(
        df_7_18["location_longitude"], df_7_18["location_latitude"]
    ),
    crs=crs_plot,
)
gdf_7_18["geometry"] = (
    gdf_7_18.to_crs(crs_norge).buffer(gdf_7_18["radius"]).to_crs(crs_plot)
)

df_7_20 = pd.read_parquet("stasjoner_med_frekvens_10_15_7_20.parquet")

gdf_7_20 = gpd.GeoDataFrame(
    df_7_20,
    geometry=gpd.points_from_xy(
        df_7_20["location_longitude"], df_7_20["location_latitude"]
    ),
    crs=crs_plot,
)

gdf_7_20["geometry"] = (
    gdf_7_20.to_crs(crs_norge).buffer(gdf_7_20["radius"]).to_crs(crs_plot)
)

In [6]:
gdf_7_18

,stopPlaceRef,route_type,name,location_longitude,location_latitude,radius,geometry
0,NSR:StopPlace:3776,Bus,Stabekk kino,10.601667,59.909478,400,"POLYGON ((10.6088 59.90972, 10.60881 59.90936,..."
1,NSR:StopPlace:5630,Bus,Nordre Gran,10.902732,59.941875,400,"POLYGON ((10.90987 59.9421, 10.90988 59.94174,..."
2,NSR:StopPlace:202,Train,Holmlia stasjon,10.797166,59.835163,500,"POLYGON ((10.80606 59.83545, 10.80607 59.83501..."
3,NSR:StopPlace:26244,Bussveien,Skadbergbakken,5.660958,58.884317,500,"POLYGON ((5.66952 58.88494, 5.66959 58.8845, 5..."
4,NSR:StopPlace:60447,Bus,Buran,10.425047,63.436397,400,"POLYGON ((10.43304 63.43665, 10.43306 63.4363,..."
...,...,...,...,...,...,...,...
1198,NSR:StopPlace:30159,Tram,Kokstadflaten,5.244435,60.291936,500,"POLYGON ((5.25335 60.2926, 5.25344 60.29216, 5..."
1199,NSR:StopPlace:31223,Bus,Li,5.346161,60.463814,400,"POLYGON ((5.35333 60.46434, 5.3534 60.46399, 5..."
1200,NSR:StopPlace:30867,Tram,Nygård,5.332619,60.384529,500,"POLYGON ((5.34156 60.38518, 5.34165 60.38475, ..."
1201,NSR:StopPlace:4327,Bus,Frogner kirke,10.707320,59.917678,400,"POLYGON ((10.71445 59.91791, 10.71446 59.91756..."


In [7]:
for loop_område, loop_gdf in områder_gdf.to_crs(crs_plot).groupby("område"):
    layers = []
    loop_sentrumsområder = gdf.sjoin(loop_gdf[["område", "geometry"]], how="inner")
    layers.append(
            pdk.Layer(
                "GeoJsonLayer",
                loop_sentrumsområder,
                opacity=0.5,
                get_fill_color=[20, 100, 200, 150],
                get_line_color=[255, 255, 255],
                pickable=True,
            )
        )

    loop_gdf_7_20 = gdf_7_20.sjoin(loop_gdf[["område", "geometry"]], how="inner")
    if loop_gdf_7_20.shape[0]>0:
        layers.append(
            pdk.Layer(
                "GeoJsonLayer",
                loop_gdf_7_20.query("(route_type=='Bus') | (route_type=='Bussveien')"),
                stroked=True,
                get_line_width=25,
                get_line_color=[255, 150, 0, 240],
                get_fill_color=[68, 79, 85, 0],
                pickable=True,
            )
        )
        layers.append(
                pdk.Layer(
                    "GeoJsonLayer",
                    loop_gdf_7_20.query("(route_type!='Bus') & (route_type!='Bussveien')"),
                    stroked=True,
                    get_line_width=25,
                    get_line_color=[68, 79, 85, 240],
                    get_fill_color=[68, 79, 85, 0],
                    pickable=True,
                )
            )
    

        initial_view = pdk.data_utils.compute_view(loop_gdf_7_20[["location_longitude", "location_latitude"]].values, view_proportion=1)
    else:
        initial_view = pdk.ViewState(longitude=10, latitude=60, zoom=6)
    r = pdk.Deck(
        layers=layers, 
        initial_view_state=initial_view, 
        api_keys={"mapbox":MAPBOX_TOKEN}, 
        views=[{"@@type": "MapView", "controller": True}],
        map_style="light",
        map_provider="mapbox"
    )
    r.to_html(f"screenshot_{loop_område}.html")